In [ ]:
import os
import re
import easyocr

def extract_text_from_file(file_path):
    """Extracts raw text from an image file using EasyOCR."""
    print("Reading image text (this might take a few seconds)...")

    # Initialize the Reader for English (adds a tiny model download on the 1st run)
    reader = easyocr.Reader(['en'], gpu=False)

    # Read the text elements
    results = reader.readtext(file_path, detail=0)

    # Combine the detected text lines into one block of text
    return " ".join(results)

def rename_delivery_note(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    print(f"Processing: {os.path.basename(file_path)}...")

    # 1. Extract text via EasyOCR
    text = extract_text_from_file(file_path)

    # 2. Match the specific delivery note format
    pattern = r"DN-IKD-0426-(\d+)"
    match = re.search(pattern, text)

    if match:
        dn_number = match.group(1)
        print(f"Found Delivery Note Number: {dn_number}")

        # 3. Perform the rename execution
        dir_name = os.path.dirname(file_path)
        ext = os.path.splitext(file_path)[1]
        new_filename = f"{dn_number}{ext}"
        new_file_path = os.path.join(dir_name, new_filename)

        try:
            os.rename(file_path, new_file_path)
            print(f"Successfully renamed to: {new_filename}\n")
        except Exception as e:
            print(f"Failed to rename file: {e}\n")

    else:
        print("Could not find the delivery note pattern in the document.")
        print("--- Extracted text snippet for debugging ---")
        print(text[:500])
        print("--------------------------------------------\n")

if __name__ == "__main__":
    # Note: EasyOCR works best out-of-the-box with common image scans (.jpg, .png).
    target_file = "/2026-05-22 4.08.46 PM.jpg"
    rename_delivery_note(target_file)

In [10]:
import easyocr
from PIL import Image
import numpy as np  # <-- Add this import
import re

def get_delivery_note_number(image_path):
    # 1. Open the image and get its dimensions
    img = Image.open(image_path)
    width, height = img.size

    # 2. Define the bounding box for the Delivery Note No. section
    left = int(width * 0.45)
    top = int(height * 0.05)
    right = int(width * 0.75)
    bottom = int(height * 0.12)

    # 3. Crop the image to just that box
    cropped_img = img.crop((left, top, right, bottom))
    # cropped_img.show()
    # 4. Convert the PIL Image into a NumPy array so EasyOCR can read it
    cropped_numpy = np.array(cropped_img) # <-- Convert here

    # 5. Pass the NumPy array to EasyOCR
    reader = easyocr.Reader(['en'], gpu=False)
    results = reader.readtext(cropped_numpy) # <-- Pass the array instead

    # 6. Reconstruct the full string from all text found in the box
    all_text_found = []
    for (bbox, text, prob) in results:
        # Remove any unwanted spaces
        clean_chunk = text.replace(" ", "").strip()
        all_text_found.append(clean_chunk)

    # Combine everything found in that box into one string
    combined_string = "".join(all_text_found)
    print(f"[DEBUG] Raw combined text: '{combined_string}'")

    # This regex captures the main DN body, and specifically isolates the
    # SHARED digit right before the month name (e.g., '-May-')
    match = re.search(r'(DN-[A-Z0-9]+-\d{4}-)(\d)-[A-Za-z]{3}-\d{2}', combined_string, re.IGNORECASE)

    if match:
        # match.group(1) is 'DN-IKD-0526-'
        # match.group(2) is the shared '1'
        full_delivery_note = match.group(1) + match.group(2)
        return full_delivery_note

    # Fallback: If the date is a double digit (e.g., 22-May-26) and doesn't merge,
    # the normal pattern will still work perfectly.
    fallback_match = re.search(r'DN-[A-Z0-9]+-\d{4}-\d+', combined_string, re.IGNORECASE)
    if fallback_match:
        return fallback_match.group(0)

    return None

# Example usage:
target_file = "../2026-05-22 4.08.46 PM.jpg"
note_number = get_delivery_note_number(target_file)

if note_number:
    print(f"Success! Found Delivery Note No: {note_number}")
else:
    print("Could not isolate the Delivery Note Number.")

Using CPU. Note: This module is much faster with a GPU.


[DEBUG] Raw combined text: 'NOTEDeliveryNoleNoDatedDN-IKD-0526-1-May-26Mode(Terms0f'
Success! Found Delivery Note No: DN-IKD-0526-1


In [12]:
import Cocoa
import Vision
import Foundation
from PIL import Image
import re

def get_delivery_note_apple_vision(image_path):
    # 1. Open and crop the image to isolate the top-right quadrant
    img = Image.open(image_path)
    width, height = img.size

    left = int(width * 0.45)
    top = int(height * 0.05)
    right = int(width * 0.75)
    bottom = int(height * 0.12)
    cropped_img = img.crop((left, top, right, bottom))

    # 2. Convert the cropped PIL image into an Apple NSData object
    # We save it to a memory buffer instead of writing it to disk
    import io
    img_byte_arr = io.BytesIO()
    cropped_img.save(img_byte_arr, format='JPEG')
    img_data = NSData = Cocoa.NSData.dataWithBytes_length_(
        img_byte_arr.getvalue(),
        len(img_byte_arr.getvalue())
    )

    # 3. Create the text recognition request
    text_strings = []

    def completion_handler(request, error):
        if error:
            print(f"Vision Error: {error}")
            return

        # Get results
        observations = request.results()
        for observation in observations:
            # Apple Vision automatically sorts layout fragments cleanly!
            text_strings.append(observation.text())

    # Configure the request handler
    request = Vision.VNRecognizeTextRequest.alloc().initWithCompletionHandler_(completion_handler)
    request.setRecognitionLevel_(Vision.VNRequestTextRecognitionLevelAccurate) # Use high accuracy mode
    request.setUsesLanguageCorrection_(True)

    # 4. Execute the request
    handler = Vision.VNImageRequestHandler.alloc().initWithData_options_(img_data, None)
    success, error = handler.performRequests_error_([request], None)

    if not success:
        print(f"Failed to perform Vision request: {error}")
        return None

    # 5. Process the extracted text fragments
    combined_string = "".join(text_strings).replace(" ", "")
    print(f"[DEBUG] Apple Vision Raw Text: '{combined_string}'")

    # This regex looks explicitly for the DN structure and captures exactly ONE digit after the final dash,
    # completely ignoring whatever digits or months follow it as part of the date.
    match = re.search(r'(DN-[A-Z0-9]+-\d{4}-\d)', combined_string, re.IGNORECASE)

    if match:
        return match.group(1)

    return None

# Run the test
target_file = "../2026-05-22 4.08.46 PM.jpg"
note_number = get_delivery_note_apple_vision(target_file)

if note_number:
    print(f"Success! Found Delivery Note No: {note_number}")
else:
    print("Could not isolate the Delivery Note Number.")

[DEBUG] Apple Vision Raw Text: 'NOTEDeliveryNoteNoDatedDN-IKD-0526-11-May-26Mode/Terms'
Success! Found Delivery Note No: DN-IKD-0526-1


In [ ]:
import os
import sys
import platform
import re
import io
from PIL import Image

# Import Textual TUI widgets
from textual.app import App, ComposeResult
from textual.containers import Container, Horizontal, Vertical
from textual.widgets import Header, Footer, Static, Button, Input, Log

# ==========================================
# 1. CROSS-PLATFORM OCR IMPLEMENTATION CORNER
# ==========================================

def crop_document_target(image_path):
    """Crops the input image to focus strictly on the top-right metadata box."""
    img = Image.open(image_path)
    width, height = img.size
    left = int(width * 0.45)
    top = int(height * 0.05)
    right = int(width * 0.75)
    bottom = int(height * 0.12)
    return img.crop((left, top, right, bottom))

def parse_cleaned_string(combined_string):
    """Regex rule tailored to extract the exact delivery note identity."""
    match = re.search(r'(DN-[A-Z0-9]+-\d{4}-\d)', combined_string, re.IGNORECASE)
    if match:
        return match.group(1)
    return None

def run_mac_ocr(cropped_img):
    """Extracts text using Apple's Native Vision Framework via PyObjC."""
    import Cocoa
    import Vision

    img_byte_arr = io.BytesIO()
    cropped_img.save(img_byte_arr, format='JPEG')
    img_data = Cocoa.NSData.dataWithBytes_length_(
        img_byte_arr.getvalue(), len(img_byte_arr.getvalue())
    )

    text_strings = []
    def completion_handler(request, error):
        if not error:
            for observation in request.results():
                text_strings.append(observation.text())

    request = Vision.VNRecognizeTextRequest.alloc().initWithCompletionHandler_(completion_handler)
    request.setRecognitionLevel_(Vision.VNRequestTextRecognitionLevelAccurate)

    handler = Vision.VNImageRequestHandler.alloc().initWithData_options_(img_data, None)
    success, _ = handler.performRequests_error_([request], None)

    if success:
        return "".join(text_strings).replace(" ", "")
    return ""

def run_windows_ocr(cropped_img):
    """Extracts text using Windows Media OCR Framework via WinRT (Asynchronous)."""
    import asyncio
    import winrt.windows.media.ocr as ocr
    import winrt.windows.graphics.imaging as imaging
    import winrt.windows.storage.streams as streams

    async def _async_win_ocr():
        img_byte_arr = io.BytesIO()
        cropped_img.save(img_byte_arr, format='JPEG')

        data_writer = streams.DataWriter()
        data_writer.write_bytes(list(img_byte_arr.getvalue()))
        stream = streams.InMemoryRandomAccessStream()
        await stream.write_async(data_writer.detach_buffer())
        stream.seek(0)

        decoder = await imaging.ImageDecoder.create_async(stream)
        software_bitmap = await decoder.get_software_bitmap_async()

        engine = ocr.OcrEngine.try_create_from_user_profile_languages()
        ocr_result = await engine.recognize_async(software_bitmap)

        lines = [line.text for line in ocr_result.lines]
        return "".join(lines).replace(" ", "")

    # Run the async Windows runtime loop inside our synchronous engine wrapper
    return asyncio.run(_async_win_ocr())

def extract_delivery_note(image_path, log_callback):
    """Detects platform architecture and triggers the native extraction sequence."""
    current_os = platform.system()
    log_callback(f"Detected Platform OS: [bold cyan]{current_os}[/bold cyan]")

    if not os.path.exists(image_path):
        return "Error: Selected File Path Not Found"

    cropped_img = crop_document_target(image_path)
    raw_text = ""

    if current_os == "Darwin":  # macOS
        log_callback("Initializing Apple Vision Engine...")
        raw_text = run_mac_ocr(cropped_img)
    elif current_os == "Windows":
        log_callback("Initializing Windows Media OCR Engine...")
        raw_text = run_windows_ocr(cropped_img)
    else:
        return "Error: Operating system platform not explicitly supported."

    log_callback(f"Raw Unified Text Token: '{raw_text}'")
    parsed_result = parse_cleaned_string(raw_text)
    return parsed_result if parsed_result else "Failed to parse matching structure."

# ==========================================
# 2. TEXTUAL TUI LAYOUT APPLICATION CORNER
# ==========================================

class OCRScannerApp(App):
    """A beautiful terminal dashboard for cross-platform OCR extraction."""
    CSS = """
    Screen {
        background: #1e1e2e;
    }
    #main-container {
        padding: 1 2;
    }
    .panel-title {
        text-style: bold;
        color: #cdd6f4;
        margin-bottom: 1;
    }
    Input {
        background: #313244;
        color: #cdd6f4;
        border: tall #cba6f7;
    }
    Button {
        background: #a6e3a1;
        color: #11111b;
        text-style: bold;
        border: none;
        margin-top: 1;
    }
    Button:hover {
        background: #94e2d5;
    }
    Log {
        background: #11111b;
        border: solid #45475a;
        min-height: 8;
        margin-top: 1;
    }
    #result-box {
        background: #45475a;
        color: #f9e2af;
        border: round #f9e2af;
        padding: 1 2;
        text-align: center;
        text-style: bold;
        margin-top: 1;
    }
    """

    BINDINGS = [("q", "quit", "Quit Application")]

    def compose(self) -> ComposeResult:
        yield Header(show_clock=True)
        with Container(id="main-container"):
            yield Static("📄 SCAN TARGET CONFIGURATION", classes="panel-title")
            yield Input(placeholder="Enter absolute image path (e.g., 2026-05-22 4.08.46 PM.jpg)", id="file-input")
            yield Button("Execute Native OCR Engine Scan", id="scan-btn", variant="success")

            yield Static("\n🖥️ ENGINE SYSTEM EXECUTION LOGS", classes="panel-title")
            yield Log(id="engine-log")

            yield Static("\n🎯 PARSED OUTPUT IDENTIFIER RESULT", classes="panel-title")
            yield Static("Awaiting document pathway initialization...", id="result-box")
        yield Footer()

    def on_button_pressed(self, event: Button.Pressed) -> None:
        if event.button.id == "scan-btn":
            input_path = self.query_one("#file-input", Input).value.strip()
            logger = self.query_one("#engine-log", Log)
            result_display = self.query_one("#result-box", Static)

            if not input_path:
                logger.write_line("[WARNING] Target filepath string is empty.")
                result_display.update("Error: Input Path Empty")
                return

            logger.clear()
            logger.write_line(f"Targeting asset: {input_path}")

            try:
                # Run the pipeline dynamically
                detected_no = extract_delivery_note(input_path, logger.write_line)
                logger.write_line(f"[SUCCESS] Operational sequence finalized.")
                result_display.update(f"DELIVERY NOTE NO: {detected_no}")
            except Exception as ex:
                logger.write_line(f"[CRITICAL ERROR] Core Execution Failed: {str(ex)}")
                result_display.update("Extraction Crash Interrupted Flow")

if __name__ == "__main__":
    OCRScannerApp().run()